# Incoorporating new files

## Setup

Add your data path here (`base_path`)

In [18]:
# Add path
import sys
sys.path.append('../')

import importlib 
from util import config
from util import utilities as util
import util.data_loader as data_loader  # <-- add
from util.data_loader import DataLoader

import os
import shutil
import pandas as pd
from tqdm import tqdm

# -------------------------
# NEW DATA ROOT (only)
# -------------------------
NEW_ROOT = "/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/new-data"

# point metadata CSVs to NEW only
config.trial_info_path  = os.path.join(NEW_ROOT, "trial_info_20260105.csv")
config.animal_info_path = os.path.join(NEW_ROOT, "animal_info_20251220.csv")

# point data home to NEW only (important if DataLoader uses config.data_path)
config.data_path = NEW_ROOT

# reload util so it reads the updated config values
importlib.reload(util)
importlib.reload(data_loader)

# now this can ONLY read the NEW csvs
metadata_new = util.load_metadata(type="combined")

print("Using:")
print(" trial_info_path :", config.trial_info_path)
print(" animal_info_path:", config.animal_info_path)
print(" data_path       :", config.data_path)
print(" rows            :", len(metadata_new))

Using:
 trial_info_path : /Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/new-data/trial_info_20260105.csv
 animal_info_path: /Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/new-data/animal_info_20251220.csv
 data_path       : /Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/new-data
 rows            : 5147


## Utilities

In [19]:
def load_new_metadata():
    # util.load_metadata reads config.animal_info_path / config.trial_info_path
    return util.load_metadata(type="combined")

base_path = NEW_ROOT

def get_date_format(date):
    ''' Convert YYYYMMDD to "YYYY-MM-DD" '''
    date_str = str(date)
    date_formatted = f"{date_str[0:4]}-{date_str[4:6]}-{date_str[6:8]}"
    return date_formatted

def generate_source_file_path(dl, data_type = 'behVideo'):
    '''
    Files for each trial:
        * trackInfo: `*.positions.csv`
        * exampleImg: `*.trace.png`
        * behVideo: `*.avi`
    Files for each day:
        * triggerLoc: `*.locations.csv`
    '''
    date_formatted = get_date_format(dl.date)
    date_folder_path = os.path.join(base_path, 'data', date_formatted)

    if data_type == 'triggerLoc':
        return os.path.join(date_folder_path, 'locations.csv')
    elif data_type == 'behVideo':
        return os.path.join(date_folder_path, 'movie', f"{date_formatted}_{dl.id}_trial_{dl.trial_id}.avi")
    elif data_type == 'trackInfo':
        return os.path.join(date_folder_path, 'movie', 'tracking', f"{date_formatted}_{dl.id}_trial_{dl.trial_id}_positions.csv")
    elif data_type == 'exampleImg':
        return os.path.join(date_folder_path, 'movie', 'tracking', f"{date_formatted}_{dl.id}_trial_{dl.trial_id}_trace.png")
    else:
        print('Unknown data type')

def generate_desti_file_path(dl, data_type = 'behVideo'):
    '''
    Files for each trial:
        * trackInfo: `*.positions.csv`
        * exampleImg: `*.trace.png`
        * behVideo: `*.avi`
    Files for each day:
        * triggerLoc: `*.locations.csv`
    '''
    if data_type == 'triggerLoc':
        return dl.generate_filepath('triggerLoc', 'rawdata', 'behav', 'csv')
    if data_type == 'trackInfo':
        return dl.generate_filepath('position', 'derivatives', 'behav', 'csv')
    if data_type == 'exampleImg':
        return dl.generate_filepath('trace', 'derivatives', 'behav', 'png')
    if data_type == 'behVideo':
        return dl.generate_filepath('video', 'rawdata', 'behav', 'avi')
    else: 
        print('Unknown data type')

def check_source_file_exists(dl, data_type):
    '''
    Files for each trial:
        * trackInfo: `*.positions.csv`
        * exampleImg: `*.trace.png`
        * behVideo: `*.avi`
    Files for each day:
        * triggerLoc: `*.locations.csv`
    '''
    file = generate_source_file_path(dl, data_type)
    if not os.path.isfile(file):
        print(f'{dl.id}-{dl.date}-trial_{dl.trial_id}: {data_type} missing')
        return False
    return True


In [20]:
metadata = load_new_metadata()

## Check existance of all files  

Files for each trial:  
    * trackInfo: `*.positions.csv`  
    * exampleImg: `*.trace.png`  
    * behVideo: `*.avi`  
Files for each day:  
    * triggerLoc: `*.locations.csv`  

In [21]:
data_types = {'trackInfo', 'exampleImg', 'behVideo', 'triggerLoc'}

flag = True
for rowIdx in range(len(metadata)):
    dl = DataLoader(metadata_ses=metadata.iloc[rowIdx])
    for data_type in data_types:
        if not check_source_file_exists(dl, data_type):
            flag = False

if flag:
    print('All source files exist!')

A-20241205-trial_1: trackInfo missing
A-20241205-trial_1: exampleImg missing
A-20241205-trial_1: behVideo missing
A-20241205-trial_1: triggerLoc missing
A-20241205-trial_2: trackInfo missing
A-20241205-trial_2: exampleImg missing
A-20241205-trial_2: behVideo missing
A-20241205-trial_2: triggerLoc missing
A-20241205-trial_3: trackInfo missing
A-20241205-trial_3: exampleImg missing
A-20241205-trial_3: behVideo missing
A-20241205-trial_3: triggerLoc missing
A-20241205-trial_4: trackInfo missing
A-20241205-trial_4: exampleImg missing
A-20241205-trial_4: behVideo missing
A-20241205-trial_4: triggerLoc missing
A-20241205-trial_5: trackInfo missing
A-20241205-trial_5: exampleImg missing
A-20241205-trial_5: behVideo missing
A-20241205-trial_5: triggerLoc missing
A-20241205-trial_6: trackInfo missing
A-20241205-trial_6: exampleImg missing
A-20241205-trial_6: behVideo missing
A-20241205-trial_6: triggerLoc missing
A-20241206-trial_1: trackInfo missing
A-20241206-trial_1: exampleImg missing
A-202

## Move data

In [22]:
for rowIdx in tqdm(range(len(metadata))):
    dl = DataLoader(metadata_ses=metadata.iloc[rowIdx])
    for data_type in data_types:
        source_file_path = generate_source_file_path(dl, data_type)
        desti_file_path = generate_desti_file_path(dl, data_type)

        os.makedirs(os.path.dirname(desti_file_path), exist_ok=True)

        if not os.path.isfile(source_file_path):
            print(f'{dl.id}-{dl.date}-trial_{dl.trial_id}: {data_type} missing')
            continue

        if data_type == 'triggerLoc':
            loc = pd.read_csv(source_file_path)
            loc = loc.rename(columns={
                'Animal': 'id',
                'Well_row': 'well_row',
                'Well_col': 'well_col',
                'Valence': 'valence',
            })
            loc = loc[loc['id'] == dl.id].reset_index(drop=True)
            loc.to_csv(desti_file_path, index=False)
        else:
            shutil.copy2(source_file_path, desti_file_path)
        

 28%|██▊       | 1447/5147 [00:00<00:00, 7323.52it/s]

A-20241205-trial_1: trackInfo missing
A-20241205-trial_1: exampleImg missing
A-20241205-trial_1: behVideo missing
A-20241205-trial_1: triggerLoc missing
A-20241205-trial_2: trackInfo missing
A-20241205-trial_2: exampleImg missing
A-20241205-trial_2: behVideo missing
A-20241205-trial_2: triggerLoc missing
A-20241205-trial_3: trackInfo missing
A-20241205-trial_3: exampleImg missing
A-20241205-trial_3: behVideo missing
A-20241205-trial_3: triggerLoc missing
A-20241205-trial_4: trackInfo missing
A-20241205-trial_4: exampleImg missing
A-20241205-trial_4: behVideo missing
A-20241205-trial_4: triggerLoc missing
A-20241205-trial_5: trackInfo missing
A-20241205-trial_5: exampleImg missing
A-20241205-trial_5: behVideo missing
A-20241205-trial_5: triggerLoc missing
A-20241205-trial_6: trackInfo missing
A-20241205-trial_6: exampleImg missing
A-20241205-trial_6: behVideo missing
A-20241205-trial_6: triggerLoc missing
A-20241206-trial_1: trackInfo missing
A-20241206-trial_1: exampleImg missing
A-202

 42%|████▏     | 2180/5147 [00:00<00:00, 6043.75it/s]

AL-20250716-trial_7: trackInfo missing
AL-20250716-trial_7: exampleImg missing
AL-20250716-trial_7: behVideo missing
AL-20250716-trial_7: triggerLoc missing
AL-20250716-trial_8: trackInfo missing
AL-20250716-trial_8: exampleImg missing
AL-20250716-trial_8: behVideo missing
AL-20250716-trial_8: triggerLoc missing
AL-20250716-trial_11: trackInfo missing
AL-20250716-trial_11: exampleImg missing
AL-20250716-trial_11: behVideo missing
AL-20250716-trial_11: triggerLoc missing
AL-20250717-trial_1: trackInfo missing
AL-20250717-trial_1: exampleImg missing
AL-20250717-trial_1: behVideo missing
AL-20250717-trial_1: triggerLoc missing
AL-20250717-trial_2: trackInfo missing
AL-20250717-trial_2: exampleImg missing
AL-20250717-trial_2: behVideo missing
AL-20250717-trial_2: triggerLoc missing
AL-20250717-trial_3: trackInfo missing
AL-20250717-trial_3: exampleImg missing
AL-20250717-trial_3: behVideo missing
AL-20250717-trial_3: triggerLoc missing
AL-20250717-trial_4: trackInfo missing
AL-20250717-tri

 70%|███████   | 3616/5147 [00:00<00:00, 6667.95it/s]

G-20241215-trial_2: behVideo missing
G-20241215-trial_2: triggerLoc missing
G-20241215-trial_3: trackInfo missing
G-20241215-trial_3: exampleImg missing
G-20241215-trial_3: behVideo missing
G-20241215-trial_3: triggerLoc missing
G-20241215-trial_4: trackInfo missing
G-20241215-trial_4: exampleImg missing
G-20241215-trial_4: behVideo missing
G-20241215-trial_4: triggerLoc missing
G-20241215-trial_5: trackInfo missing
G-20241215-trial_5: exampleImg missing
G-20241215-trial_5: behVideo missing
G-20241215-trial_5: triggerLoc missing
G-20241215-trial_6: trackInfo missing
G-20241215-trial_6: exampleImg missing
G-20241215-trial_6: behVideo missing
G-20241215-trial_6: triggerLoc missing
G-20241215-trial_7: trackInfo missing
G-20241215-trial_7: exampleImg missing
G-20241215-trial_7: behVideo missing
G-20241215-trial_7: triggerLoc missing
G-20241215-trial_8: trackInfo missing
G-20241215-trial_8: exampleImg missing
G-20241215-trial_8: behVideo missing
G-20241215-trial_8: triggerLoc missing
G-2024

 85%|████████▌ | 4385/5147 [00:00<00:00, 6995.59it/s]

AV-20250812-trial_1: trackInfo missing
AV-20250812-trial_1: exampleImg missing
AV-20250812-trial_1: behVideo missing
AV-20250812-trial_1: triggerLoc missing
AV-20250812-trial_2: trackInfo missing
AV-20250812-trial_2: exampleImg missing
AV-20250812-trial_2: behVideo missing
AV-20250812-trial_2: triggerLoc missing
AV-20250812-trial_3: trackInfo missing
AV-20250812-trial_3: exampleImg missing
AV-20250812-trial_3: behVideo missing
AV-20250812-trial_3: triggerLoc missing
AV-20250812-trial_4: trackInfo missing
AV-20250812-trial_4: exampleImg missing
AV-20250812-trial_4: behVideo missing
AV-20250812-trial_4: triggerLoc missing
AV-20250812-trial_5: trackInfo missing
AV-20250812-trial_5: exampleImg missing
AV-20250812-trial_5: behVideo missing
AV-20250812-trial_5: triggerLoc missing
AV-20250812-trial_6: trackInfo missing
AV-20250812-trial_6: exampleImg missing
AV-20250812-trial_6: behVideo missing
AV-20250812-trial_6: triggerLoc missing
AV-20250812-trial_7: trackInfo missing
AV-20250812-trial_7

100%|██████████| 5147/5147 [00:05<00:00, 944.26it/s] 
